# Quick Start: MIDAS Regression in 10 Lines

**Estimated time:** 2 minutes

Run this notebook to get a working MIDAS regression immediately. No prior reading required.

**What this does:**
1. Loads quarterly GDP growth and monthly IP growth (CSV fallback)
2. Builds the MIDAS data matrix (aligns mixed frequencies)
3. Estimates Beta MIDAS by profile NLS
4. Plots the weight function and fit

To go deeper: start with Module 01 notebooks.

In [ ]:
# ============================================================
# MIDAS Quick Start
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.stats import beta as beta_dist
import os, warnings
warnings.filterwarnings('ignore')

# --- 1. Load data ---
def find_csv(filename):
    for base in ['../modules/module_00_foundations/resources',
                 'modules/module_00_foundations/resources']:
        path = os.path.join(base, filename)
        if os.path.exists(path):
            return pd.read_csv(path, index_col='date', parse_dates=True).squeeze()
    raise FileNotFoundError(filename)

gdp_growth = find_csv('gdp_quarterly.csv')
ip_growth  = find_csv('industrial_production_monthly.csv')
print(f"GDP: {len(gdp_growth)} quarters | IP: {len(ip_growth)} months")

# --- 2. Build MIDAS matrix (K=12 monthly lags) ---
K = 12
y_q = gdp_growth.copy(); y_q.index = gdp_growth.index.to_period('Q')
x_m = ip_growth.copy();  x_m.index = ip_growth.index.to_period('M')

rows_Y, rows_X = [], []
for q in y_q.index:
    last = q.asfreq('M', how='end')
    lags = [last - i for i in range(K)]
    if any(l not in x_m.index for l in lags): continue
    rows_Y.append(y_q[q]); rows_X.append([x_m[l] for l in lags])

Y = np.array(rows_Y); X = np.array(rows_X)
print(f"MIDAS dataset: T={len(Y)}, K={K}")

# --- 3. Profile NLS estimation ---
def beta_w(K, t1, t2):
    x = (np.arange(K) + 0.5) / K
    raw = beta_dist.pdf(1 - x, max(t1, 0.01), max(t2, 0.01))
    return raw / raw.sum() if raw.sum() > 1e-12 else np.ones(K)/K

def psse(theta):
    t1, t2 = theta
    if t1 <= 0.01 or t2 <= 0.01: return 1e10
    xw = X @ beta_w(K, t1, t2)
    xc, yc = xw - xw.mean(), Y - Y.mean()
    ss = xc @ xc
    if ss < 1e-12: return np.sum((Y - Y.mean())**2)
    b = yc @ xc / ss; a = Y.mean() - b * xw.mean()
    return np.sum((Y - a - b * xw)**2)

best = min([minimize(psse, t0, method='Nelder-Mead',
                     options={'maxiter': 20000, 'xatol': 1e-8})
            for t0 in [(1.,5.), (1.5,4.), (2.,3.), (1.,1.)]],
           key=lambda r: r.fun)

t1, t2 = max(best.x[0], 0.01), max(best.x[1], 0.01)
w = beta_w(K, t1, t2); xw = X @ w
xc, yc = xw - xw.mean(), Y - Y.mean()
beta_hat = yc @ xc / (xc @ xc)
alpha_hat = Y.mean() - beta_hat * xw.mean()
r2 = 1 - psse([t1, t2]) / np.sum((Y - Y.mean())**2)

print(f"\nResults:")
print(f"  alpha  = {alpha_hat:.4f}")
print(f"  beta   = {beta_hat:.4f}")
print(f"  theta1 = {t1:.4f}")
print(f"  theta2 = {t2:.4f}")
print(f"  R²     = {r2:.4f}")

# --- 4. Plot ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(range(K), w, color='steelblue', alpha=0.8)
axes[0].axhline(1/K, color='tomato', linestyle='--', label='Equal weight')
axes[0].set_title(f'Beta MIDAS Weights (θ₁={t1:.2f}, θ₂={t2:.2f})')
axes[0].set_xlabel('Lag j (j=0 = most recent month)')
axes[0].set_ylabel('Weight'); axes[0].legend()

fitted = alpha_hat + beta_hat * xw
axes[1].scatter(fitted, Y, s=25, alpha=0.7, color='steelblue')
mn, mx = min(fitted.min(), Y.min()), max(fitted.max(), Y.max())
axes[1].plot([mn, mx], [mn, mx], 'r--', linewidth=1.5, label='45° line')
axes[1].set_xlabel('Fitted GDP growth (%)')
axes[1].set_ylabel('Actual GDP growth (%)')
axes[1].set_title(f'Fitted vs Actual (R²={r2:.3f})')
axes[1].legend()

plt.suptitle('MIDAS Quick Start: GDP ~ IP', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()
print("\nQuick start complete. Explore Module 01 for full details.")